# 09 — Perplexidade contra uma régua medida + MoE

Duas ferramentas de medição do laboratório em um caderno:

1. **Perplexidade com baseline**: números absolutos de perda não significam nada sem uma
   régua. A régua mínima honesta é o classificador **unigrama** (frequência de tokens no
   trem) — um modelo que não o supera não aprendeu estrutura, decorou frequências.
2. **MoE e custo real por token**: o núcleo suporta FFN esparsa; medimos parâmetros
   *totais* versus *ativos por token* e o mapa de uso dos especialistas.

Pré-requisito: caderno 01 executado (carrega `artifacts/modelo_puro`).


In [ ]:
from pathlib import Path
import math
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from labia.trainer.dados import dividir_corpus

MODEL_DIR = Path('artifacts/modelo_puro')
if not (MODEL_DIR / 'config.json').is_file():
    raise FileNotFoundError('Modelo ausente. Execute primeiro o caderno 01 até a célula de salvamento.')

disp = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
modelo = AutoModelForCausalLM.from_pretrained(MODEL_DIR).to(disp).eval()

corpus = Path('data/corpus.txt').read_text(encoding='utf-8')
trem_texto, val_texto = dividir_corpus(corpus, semente=42)
BLOCK = modelo.config.n_positions
print(f'contexto do modelo: {BLOCK} | dispositivo: {disp}')


## 1. Baseline unigrama (a régua)

`CE_unigrama = média(-log p(token))` no conjunto de validação, com suavização.
Perplexidade é `exp(CE)` — leia como "quantos candidatos, em média, o modelo considera
por token".

In [ ]:
trem_ids = tokenizer(trem_texto, add_special_tokens=False)['input_ids']
val_ids = tokenizer(val_texto, add_special_tokens=False)['input_ids']

contagem = Counter(trem_ids)
V = len(tokenizer)
alfa = 1.0
def log_p(tok_id: int) -> float:
    return math.log((contagem.get(tok_id, 0) + alfa) / (sum(contagem.values()) + alfa * V))

ce_uni = -sum(log_p(t) for t in val_ids) / len(val_ids)
print(f'unigrama: CE {ce_uni:.3f} | perplexidade {math.exp(ce_uni):.1f} | vocab {V} | tokens no val {len(val_ids)}')


## 2. Perplexidade do modelo no mesmo conjunto de validação

In [ ]:
# GPT2LMHeadModel já desloca os alvos internamente quando recebe `labels`.
@torch.no_grad()
def ce_modelo(ids):
    blocos = [ids[i:i + BLOCK] for i in range(0, len(ids) - 1, BLOCK)]
    total, n_tok = 0.0, 0
    for b in blocos:
        if len(b) < 2:
            continue
        entrada = torch.tensor([b], device=disp)
        saida = modelo(entrada, labels=entrada)
        total += float(saida.loss) * len(b)
        n_tok += len(b)
    return total / max(1, n_tok)

ce_mod = ce_modelo(val_ids)
ppl_mod = math.exp(ce_mod)
ppl_uni = math.exp(ce_uni)
print(f'modelo:    CE {ce_mod:.3f} | perplexidade {ppl_mod:.1f}')
print(f'unigrama:  CE {ce_uni:.3f} | perplexidade {ppl_uni:.1f}')
veredito = 'superou a régua' if ppl_mod < ppl_uni else 'NÃO superou a régua — mais treino ou mais dados'
print(f'relação: {ppl_uni / ppl_mod:.2f}× menos incerteza que o unigrama → {veredito}')


Repare que a comparação com uma métrica **absoluta** seria sem sentido em um corpus de
centenas de linhas repetidas; o que vale é a posição relativa à régua medida no mesmo
dados. Isso é o que o projeto usa como barra nas metas de treino.

## 3. MoE: tamanho ≠ custo por token

Um FFN MoE com *n* especialistas guardando *n* pesos carrega só *k* por token.

In [ ]:
from labia.models.gpt import ConfigGPT, GPT
from labia.trainer.dados import montar_dataset
from tokenizers import Tokenizer

tok_puro = Tokenizer.from_file('artifacts/tokenizer_puro/tokenizer.json')
torch.manual_seed(42)
cfg_moe = ConfigGPT(vocab=tok_puro.get_vocab_size(), dim=128, camadas=2, cabecas=4,
                    janela_ctx=128, abandono=0.0, n_especialistas=4, top_k=1,
                    coef_auxiliar=0.01, norm='rmsnorm', pos='rope')
moe = GPT(cfg_moe).to(disp)
moe.init_pesos(semente=42)

x, y = montar_dataset(tok_puro, trem_texto, 128, stride=64)
lote = x[:16].to(disp)
moe.eval()
with torch.no_grad():
    moe(lote)
print('especialistas: 4 | top-k: 1')
print(f'parâmetros totais: {moe.contar_parametros():,} | ativos por token: {moe.contar_parametros_ativos():,}'
      f' ({moe.contar_parametros_ativos() / moe.contar_parametros():.0%})')
print('mapa de uso (recém-inicializado):', moe.mapa_uso_especialistas())


In [ ]:
# Um pouco de treino: o roteador se equilibra com a perda auxiliar em ação.
otim = torch.optim.AdamW(moe.parameters(), lr=5e-4)
moe.train()
for passo in range(120):
    ini = (passo * 16) % max(1, len(x) - 16)
    _, perda = moe(x[ini:ini + 16].to(disp), y[ini:ini + 16].to(disp))
    otim.zero_grad(set_to_none=True)
    perda.backward()
    otim.step()
moe.eval()
with torch.no_grad():
    moe(lote)
print('aux do roteador no último passo:', moe.ultimo_aux)
print('mapa de uso após treino:', moe.mapa_uso_especialistas())


## Exercícios

1. Zere `coef_auxiliar` e treine de novo: o mapa degrada para "um especialista come tudo"?
2. Troque `top_k` para 2 e refaça a conta de parâmetros ativos.
3. Aplique este caderno ao modelo do caderno 02 (Qwen refinado) trocando o `MODEL_DIR`:
   a régua unigrama muda com o tokenizer — o que acontece com a perplexidade relativa?